In [ ]:
import polars as pl

In [ ]:
csv_path: str = "../data/bins_info.csv"
pl.read_csv(csv_path).to_dicts()


In [ ]:
govuk_council_urls = "https://govuk-app-assets-production.s3.eu-west-1.amazonaws.com/data/local-links-manager/links_to_services_provided_by_local_authorities.csv"
la_urls = (
    pl.read_csv(govuk_council_urls)
    .select(["Authority Name", "GSS", "Description", "URL"])
    .filter(
        pl.col("Description").str.contains(
            "Household waste collection: Providing information"
        )
    )
)

In [ ]:
postcodes = (
    pl.read_csv("../NSPL_MAY_2025/Data/NSPL_MAY_2025_UK.csv")
    .select(["pcd", "laua"])
    .group_by("laua")
    .agg(pl.col("pcd").sample(3).alias("sampled_pcds"))
    .with_columns(
        [
            pl.col("sampled_pcds").list.get(0).alias("pcd1"),
            pl.col("sampled_pcds").list.get(1).alias("pcd2"),
            pl.col("sampled_pcds").list.get(2).alias("pcd3"),
        ]
    )
    .drop("sampled_pcds")
)


In [ ]:
joined_df = la_urls.join(postcodes, how="left", left_on="GSS", right_on="laua").select(
    ["Authority Name", "URL", "pcd1", "pcd2", "pcd3"]
)
joined_df.write_csv("bins_info.csv")